In [1]:
import time
from typing import Iterator

import numpy
from smodels.decomposition.theorySMS import TheorySMS
from smodels.base.genericSMS import GenericSMS
from smodels.decomposition.topologyDict import TopologyDict
from smodels.base.particleNode import ParticleNode
from smodels.base.physicsUnits import fb, GeV
from smodels.decomposition.exceptions import SModelSDecompositionError as SModelSError
from smodels.base.smodelsLogging import logger
from itertools import product
from smodels.base import runtime
from smodels.decomposition import decomposer
from smodels.base.physicsUnits import fb, GeV, TeV
from smodels.matching.theoryPrediction import theoryPredictionsFor,TheoryPredictionsCombiner
from smodels.experiment.databaseObj import Database
from smodels.base.smodelsLogging import setLogLevel
from smodels.tools.particlesLoader import load
from smodels.share.models.SMparticles import SMList
from smodels.base.model import Model
import itertools
import time
import numpy as np
setLogLevel("info")

## Load Model

In [2]:
# Load the BSM model
runtime.modelFile = "nmssmPoints/000.slha"
BSMList = load()
model = Model(BSMparticles=BSMList, SMparticles=SMList)
slhafile = 'nmssmPoints/022.slha'
model.updateParticles(inputFile=slhafile,ignorePromptQNumbers = ['eCharge','spin'])


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


In [3]:
def build_subtree_cache(particle, memo=None, visiting=None, minBR=0.0):
    """Return all weighted TheorySMS subtrees rooted at `particle` and cache by particle."""

    if memo is None:
        memo = {}
    if visiting is None:
        visiting = set()
    if particle in memo:
        return memo[particle], memo

    if particle in visiting:
        raise ValueError(
            f"Cycle detected at particle {particle}; this DFS cache expects a DAG/tree-like decay graph."
        )

    logger.debug(f"Building subtree cache for particle {particle}")
    visiting.add(particle)
    subtrees = []
    decays = []
    if hasattr(particle, "decays") and particle.decays is not None:
        decays = [decay for decay in particle.decays if decay is not None]

    if not decays:
        sms = TheorySMS()
        sms.add_node(ParticleNode(particle=particle))
        sms.decayBRs = 1.0
        subtrees.append(sms)
    else:
        for decay in decays:
            br = decay.br
            if br < minBR:
                continue
            daughters = decay.daughters
            daughter_choices = []
            for daughter in daughters:
                daughter_subtrees, memo = build_subtree_cache(daughter, memo, visiting, minBR)
                daughter_choices.append(daughter_subtrees)

            for combo in product(*daughter_choices):
                sms = TheorySMS()
                root_index = sms.add_node(ParticleNode(particle=particle))

                all_weight = br
                for daughter_sms in combo:
                    index_map = {}
                    for node_index in daughter_sms.nodeIndices:
                        node = daughter_sms.indexToNode(node_index)
                        node_copy = node.copy() if hasattr(node, "copy") else node
                        new_index = sms.add_node(node_copy)
                        index_map[node_index] = new_index

                    for edge_a, edge_b in daughter_sms.edgeIndices:
                        sms.add_edge(index_map[edge_a], index_map[edge_b])

                    sms.add_edge(root_index, index_map[daughter_sms.rootIndex])
                    all_weight *= daughter_sms.decayBRs
                    if all_weight < minBR:
                        all_weight = 0.0
                        break
                if all_weight >= minBR:
                    sms.decayBRs = all_weight
                    subtrees.append(sms)

    visiting.remove(particle)
    if any(decay is None for decay in getattr(particle, "decays", []) or []):
        sms = TheorySMS()
        sms.add_node(ParticleNode(particle=particle))
        sms.decayBRs = 1.0
        subtrees.append(sms)
    memo[particle] = tuple(subtrees)
    return memo[particle], memo



In [4]:

def decomposeNew(model, sigmacut, massCompress, invisibleCompress, minmassgap, minmassgapISR):

    xSectionList = model.xsections
    sigmacutFB = sigmacut.asNumber(fb)  # sigmacut in fb (faster comparison)

    xSectionList.removeLowerOrder()
    # Order xsections by highest xsec value to improve performance
    xSectionList.sort()

    productionSMS = []
    smsTopDict = TopologyDict()
    for pdgs in xSectionList.getPIDpairs():
        weight = xSectionList.getXsecsFor(pdgs)
        maxWeight = weight.getMaxXsec().asNumber(fb)
        if maxWeight < sigmacutFB:
            continue
        pv = ParticleNode(model.getParticle(label='PV'))
        primaryMothers = [ParticleNode(model.getParticle(pdg=pdg)) for pdg in pdgs]
        newSMS = TheorySMS()
        newSMS.maxWeight = maxWeight
        newSMS.prodXSec = weight
        pvIndex = newSMS.add_node(pv)
        motherIndices = newSMS.add_nodes_from(primaryMothers)
        newSMS.add_edges_from(product([pvIndex], motherIndices))
        productionSMS.append(newSMS)

    maxXsec = max(sms.maxWeight for sms in productionSMS)
    minBR = sigmacutFB / maxXsec
    cache = {}
    for sms in productionSMS:
        for particleNode in sms.daughters(sms.rootIndex):
            if particleNode.particle is None:
                continue
            if particleNode.particle in cache:
                continue
            subtrees, cache = build_subtree_cache(particleNode.particle, memo=cache, minBR=minBR)

    for sms in productionSMS:
        all_subtrees = [
            cache.get(sms.indexToNode(daughterIndex).particle, [])
            for daughterIndex in sms.daughterIndices(sms.rootIndex)
        ]
        daughterIndices = list(sms.daughterIndices(sms.rootIndex))
        for primary_subtrees in itertools.product(*all_subtrees):
            totalBR = 1.0
            for subtree in primary_subtrees:
                totalBR *= subtree.decayBRs
            if sms.maxWeight * totalBR < sigmacutFB:
                continue

            smsDecayed = sms.copy()
            for idaughter, subtree in enumerate(primary_subtrees):
                old2newIndexMapping = {0: daughterIndices[idaughter]}
                for nodeIndex in subtree.nodeIndices:
                    # Skip subtree root: the primary mother is already present in smsDecayed.
                    if nodeIndex == subtree.rootIndex:
                        continue
                    node = subtree.indexToNode(nodeIndex)
                    newIndex = smsDecayed.add_node(node)
                    old2newIndexMapping[nodeIndex] = newIndex

                for edgeA, edgeB in subtree.edgeIndices:
                    smsDecayed.add_edge(old2newIndexMapping[edgeA], old2newIndexMapping[edgeB])

            smsDecayed.decayBRs = totalBR
            smsDecayed.maxWeight = sms.maxWeight * totalBR
            smsDecayed.setGlobalProperties()  # Set global properties for each tree
            smsDecayed.ancestors = [smsDecayed]  # Set ancestors (before compression)
            smsTopDict.addSMS(smsDecayed)



    if massCompress or invisibleCompress:
        smsTopDict.compress(massCompress, invisibleCompress, 
                            minmassgap, minmassgapISR)

    return smsTopDict

In [5]:
massCompress = True
invisibleCompress = True

# Set main options for decomposition
sigmacut = 0.05*fb
mingap = 10.*GeV
mingapISR = 10.0*GeV


In [6]:
# Decompose model
t0 = time.time()
topDictNew = decomposeNew(model, sigmacut,
                               massCompress=massCompress, invisibleCompress=invisibleCompress,
                               minmassgap=mingap, minmassgapISR=mingapISR)


print(f'New done in {time.time()-t0:.1f} seconds, resulting in {len(topDictNew)} SMS and {len(topDictNew.getSMSList())} distinct SMS after compression.')
t0 = time.time()
topDict = decomposer.decompose(model, sigmacut,
                               massCompress=massCompress, invisibleCompress=invisibleCompress,
                               minmassgap=mingap, minmassgapISR=mingapISR)
print(f'Old done in {time.time()-t0:.1f} seconds, resulting in {len(topDict)} SMS and {len(topDict.getSMSList())} distinct SMS after compression.')



New done in 5.3 seconds, resulting in 26 SMS and 4731 distinct SMS after compression.
Old done in 4.7 seconds, resulting in 26 SMS and 4734 distinct SMS after compression.


In [7]:
# Decompose model
t0 = time.time()
topDict = decomposer.decompose(model, sigmacut,
                               massCompress=massCompress, invisibleCompress=invisibleCompress,
                               minmassgap=mingap, minmassgapISR=mingapISR)
print(f'Old done in {time.time()-t0:.1f} seconds, resulting in {len(topDict)} SMS and {len(topDict.getSMSList())} distinct SMS after compression.')
t0 = time.time()
topDictNew = decomposeNew(model, sigmacut,
                               massCompress=massCompress, invisibleCompress=invisibleCompress,
                               minmassgap=mingap, minmassgapISR=mingapISR)


print(f'New done in {time.time()-t0:.1f} seconds, resulting in {len(topDictNew)} SMS and {len(topDictNew.getSMSList())} distinct SMS after compression.')

Old done in 4.5 seconds, resulting in 26 SMS and 4733 distinct SMS after compression.
New done in 5.6 seconds, resulting in 26 SMS and 4733 distinct SMS after compression.


In [8]:
import cProfile
import pstats
import io
import time


def _profile_decomposition(fn, label, *args, **kwargs):
    profiler = cProfile.Profile()
    t0 = time.perf_counter()
    profiler.enable()
    result = fn(*args, **kwargs)
    profiler.disable()
    dt = time.perf_counter() - t0

    stats_stream = io.StringIO()
    stats = pstats.Stats(profiler, stream=stats_stream).sort_stats("cumulative")
    stats.print_stats(25)

    print(f"\n=== {label} ===")
    print(f"wall_time_s: {dt:.3f}")
    print(f"n_topologies: {len(result)}")
    print(f"n_sms: {len(result.getSMSList())}")
    print("top cumulative calls:")
    print(stats_stream.getvalue())

    return result, profiler, dt


args = (model, sigmacut)
kwargs = dict(
    massCompress=massCompress,
    invisibleCompress=invisibleCompress,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
)


res_new, prof_new, dt_new = _profile_decomposition(
    decomposeNew,
    "decomposeNew",
    *args,
    **kwargs,
)


res_old, prof_old, dt_old = _profile_decomposition(
    decomposer.decompose,
    "decomposer.decompose",
    *args,
    **kwargs,
)

print("\nSpeed ratio (new/old):", dt_new / dt_old if dt_old > 0 else float("inf"))


=== decomposeNew ===
wall_time_s: 12.801
n_topologies: 26
n_sms: 4733
top cumulative calls:
         24581372 function calls (23872252 primitive calls) in 12.272 seconds

   Ordered by: cumulative time
   List reduced from 311 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.012    0.012    6.732    6.732 /home/lessa/smodels-main/smodels/decomposition/topologyDict.py:82(compress)
     2858    0.045    0.000    5.482    0.002 /home/lessa/smodels-main/smodels/decomposition/theorySMS.py:331(compress)
    14605    0.032    0.000    3.688    0.000 /home/lessa/smodels-main/smodels/decomposition/theorySMS.py:98(setGlobalProperties)
     9231    0.095    0.000    3.382    0.000 /home/lessa/smodels-main/smodels/decomposition/topologyDict.py:25(addSMS)
    13290    0.216    0.000    2.899    0.000 /home/lessa/smodels-main/smodels/decomposition/theorySMS.py:481(invisibleCompress)
302610/176945    0.373    0.000    2.788    0.00

In [9]:
def _cumtime_for(stats_obj, needle):
    total = 0.0
    for (filename, _lineno, funcname), stat in stats_obj.stats.items():
        if needle in f"{filename}:{funcname}":
            total += stat[3]
    return total

stats_old = pstats.Stats(prof_old)
stats_new = pstats.Stats(prof_new)
hotspots = [
    "theorySMS.py:compress",
    "theorySMS.py:invisibleCompress",
    "particle.py:__new__",
    "particle.py:getinstances",
    "topologyDict.py:addSMS",
    "build_subtree_cache",
    "iterCascadeDecay",
]

print("Hotspot cumulative seconds (old/new):")
for h in hotspots:
    t_old = _cumtime_for(stats_old, h)
    t_new = _cumtime_for(stats_new, h)
    print(f"{h:40s} {t_old:8.3f} {t_new:8.3f}")

Hotspot cumulative seconds (old/new):
theorySMS.py:compress                       5.530    5.482
theorySMS.py:invisibleCompress              2.950    2.899
particle.py:__new__                         1.588    1.405
particle.py:getinstances                    0.252    0.252
topologyDict.py:addSMS                      3.623    3.382
build_subtree_cache                         0.000    0.535
iterCascadeDecay                            0.962    0.000


In [10]:
stats_new = pstats.Stats(prof_new)

new_keys = [
    key for key in stats_new.stats.keys()
    if "smodels/base/particle.py" in key[0] and key[2] == "__new__"
]

print("All __new__ entries in particle.py:")
for key in sorted(new_keys, key=lambda k: k[1]):
    cc, nc, tt, ct, callers = stats_new.stats[key]
    print(f"key={key} ncalls={nc} tottime={tt:.3f} cumtime={ct:.3f}")


def _top_callers_for_key(stats_obj, target_key, topn=8):
    callers = stats_obj.stats[target_key][4]
    rows = []
    for caller_key, caller_info in callers.items():
        cc, nc, tt, ct = caller_info[:4]
        rows.append((ct, nc, tt, cc, caller_key))
    rows.sort(reverse=True, key=lambda x: x[0])
    return rows[:topn]

for key in sorted(new_keys, key=lambda k: k[1]):
    print(f"\nTop callers for {key}:")
    for ct, nc, tt, cc, ckey in _top_callers_for_key(stats_new, key):
        print(f"caller={ckey} ncalls={nc} cumtime={ct:.3f} tottime={tt:.3f} prim_calls={cc}")

All __new__ entries in particle.py:
key=('/home/lessa/smodels-main/smodels/base/particle.py', 22, '__new__') ncalls=7028 tottime=0.084 cumtime=0.405
key=('/home/lessa/smodels-main/smodels/base/particle.py', 386, '__new__') ncalls=7028 tottime=0.016 cumtime=0.425
key=('/home/lessa/smodels-main/smodels/base/particle.py', 422, '__new__') ncalls=1912 tottime=0.151 cumtime=0.576

Top callers for ('/home/lessa/smodels-main/smodels/base/particle.py', 22, '__new__'):
caller=('/home/lessa/smodels-main/smodels/base/particle.py', 386, '__new__') ncalls=7028 cumtime=0.405 tottime=0.084 prim_calls=7028

Top callers for ('/home/lessa/smodels-main/smodels/base/particle.py', 386, '__new__'):
caller=('/home/lessa/smodels-main/smodels/decomposition/theorySMS.py', 481, 'invisibleCompress') ncalls=7028 cumtime=0.425 tottime=0.016 prim_calls=7028

Top callers for ('/home/lessa/smodels-main/smodels/base/particle.py', 422, '__new__'):
caller=('/home/lessa/smodels-main/smodels/base/particle.py', 176, '__add__

In [11]:
import gc
import random
from time import perf_counter
from smodels.base.particle import Particle


def build_fresh_model():
    runtime.modelFile = "nmssmPoints/022.slha"
    bsm_list = load()
    fresh_model = Model(BSMparticles=bsm_list, SMparticles=SMList)
    fresh_model.updateParticles(inputFile=slhafile, ignorePromptQNumbers=["eCharge", "spin"])
    return fresh_model


def reset_particle_registry():
    # Particle.__new__ uses a global weakref registry; reset it to avoid cross-run carryover.
    Particle._instances = set()
    Particle._lastID = 0


def run_single(method):
    reset_particle_registry()
    gc.collect()
    fresh_model = build_fresh_model()

    t0 = perf_counter()
    out = method(
        fresh_model,
        sigmacut,
        massCompress=massCompress,
        invisibleCompress=invisibleCompress,
        minmassgap=mingap,
        minmassgapISR=mingapISR,
    )
    dt = perf_counter() - t0
    return dt, len(out), len(out.getSMSList())


def benchmark_pair(method_a, method_b, name_a, name_b, n_reps=3, warmup=1, seed=42):
    rng = random.Random(seed)

    # Warm-up runs are not recorded.
    for _ in range(warmup):
        run_single(method_a)
        run_single(method_b)

    timings = {name_a: [], name_b: []}
    shapes = {name_a: [], name_b: []}

    for irep in range(n_reps):
        order = [(name_a, method_a), (name_b, method_b)]
        rng.shuffle(order)
        print(f"rep {irep + 1}/{n_reps} order = {[name for name, _ in order]}")

        for name, method in order:
            dt, ntop, nsms = run_single(method)
            timings[name].append(dt)
            shapes[name].append((ntop, nsms))
            print(f"  {name:16s} time={dt:8.3f}s topologies={ntop:3d} sms={nsms:4d}")

    print("\nSummary:")
    for name in [name_a, name_b]:
        vals = timings[name]
        print(
            f"{name:16s} median={np.median(vals):8.3f}s "
            f"mean={np.mean(vals):8.3f}s min={min(vals):8.3f}s max={max(vals):8.3f}s"
        )

    ratio = np.mean(timings[name_b]) / np.mean(timings[name_a])
    print(f"\nMedian ratio {name_b}/{name_a} = {ratio:.3f}")

    return timings, shapes

In [12]:

# Example usage (can take several minutes depending on sigmacut and n_reps):
timings_old_new, shapes_old_new = benchmark_pair(
    decomposer.decompose,
    decomposeNew,
    "decompose",
    "decomposeNew",
    n_reps=3,
    warmup=1,
)

INFO in model.updateParticles() in 428: Loaded 58 BSM particles
INFO in model.updateParticles() in 428: Loaded 58 BSM particles
INFO in model.updateParticles() in 428: Loaded 58 BSM particles


rep 1/3 order = ['decomposeNew', 'decompose']


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


  decomposeNew     time=   6.173s topologies= 26 sms=4851


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


...

  decomposeNew     time=   5.896s topologies= 26 sms=4851


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


  decompose        time=   4.334s topologies= 26 sms=4758
rep 3/3 order = ['decompose', 'decomposeNew']


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


  decompose        time=   4.272s topologies= 26 sms=4758
  decomposeNew     time=   5.386s topologies= 26 sms=4851

Summary:
decompose        median=   4.334s mean=   4.387s min=   4.272s max=   4.554s
decomposeNew     median=   5.896s mean=   5.818s min=   5.386s max=   6.173s

Median ratio decomposeNew/decompose = 1.326


In [13]:
# Diagnostic: check whether mismatch appears only when compression is enabled.
kwargs_nocomp = dict(
    massCompress=False,
    invisibleCompress=False,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
)

top_old_nocomp = decomposer.decompose(model, sigmacut, **kwargs_nocomp)
top_new_nocomp = decomposeNew(model, sigmacut, **kwargs_nocomp)

print("No-compression counts:", len(top_old_nocomp), len(top_new_nocomp))
print("No-compression SMS:", len(top_old_nocomp.getSMSList()), len(top_new_nocomp.getSMSList()))

No-compression counts: 24 24
No-compression SMS: 2858 3079


In [14]:
# Diagnostic: does removing _isStable/_isPrompt from normalization recover old/new agreement?
from smodels.base.particle import Particle, InvisibleParticle


def _norm_without_runtime_flags(cls, data):
    d = dict(data.items())
    d.pop('_id', None)
    d.pop('_comp', None)
    d.pop('_isStable', None)
    d.pop('_isPrompt', None)
    return d

Particle._normalizedAttrDict = classmethod(_norm_without_runtime_flags)

# Reset registries so both runs start from the same particle object state.
Particle._instances = set(); Particle._lastID = 0
InvisibleParticle._instances = set(); InvisibleParticle._lastID = 0

old_fix = decomposer.decompose(
    model,
    sigmacut,
    massCompress=massCompress,
    invisibleCompress=invisibleCompress,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
)

new_fix = decomposeNew(
    model,
    sigmacut,
    massCompress=massCompress,
    invisibleCompress=invisibleCompress,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
)

print("After normalization fix:")
print("topologies:", len(old_fix), len(new_fix))
print("sms:", len(old_fix.getSMSList()), len(new_fix.getSMSList()))

After normalization fix:
topologies: 26 26
sms: 4932 4981


In [15]:
# Fair mismatch check: fresh model and fresh particle registries for each method run.
from smodels.base.particle import Particle, InvisibleParticle


def _fresh_model():
    runtime.modelFile = "nmssmPoints/000.slha"
    bsm = load()
    m = Model(BSMparticles=bsm, SMparticles=SMList)
    m.updateParticles(inputFile=slhafile, ignorePromptQNumbers=['eCharge', 'spin'])
    return m


def _run_once(method):
    Particle._instances = set(); Particle._lastID = 0
    InvisibleParticle._instances = set(); InvisibleParticle._lastID = 0
    m = _fresh_model()
    return method(
        m,
        sigmacut,
        massCompress=massCompress,
        invisibleCompress=invisibleCompress,
        minmassgap=mingap,
        minmassgapISR=mingapISR,
    )


def _set_norm(remove_runtime_flags):
    def _norm(cls, data):
        d = dict(data.items())
        d.pop('_id', None)
        d.pop('_comp', None)
        if remove_runtime_flags:
            d.pop('_isStable', None)
            d.pop('_isPrompt', None)
        return d
    Particle._normalizedAttrDict = classmethod(_norm)

for remove_flags in [False, True]:
    _set_norm(remove_flags)
    old_res = _run_once(decomposer.decompose)
    new_res = _run_once(decomposeNew)
    print(f"remove_runtime_flags={remove_flags}")
    print("  topologies:", len(old_res), len(new_res))
    print("  sms:", len(old_res.getSMSList()), len(new_res.getSMSList()))

INFO in model.updateParticles() in 428: Loaded 58 BSM particles
INFO in model.updateParticles() in 428: Loaded 58 BSM particles
INFO in model.updateParticles() in 428: Loaded 58 BSM particles


remove_runtime_flags=False
  topologies: 26 26
  sms: 4651 4651


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


remove_runtime_flags=True
  topologies: 26 26
  sms: 4722 4722
